# 03 — Kalibrasi Terpisah `β*` (Validation 20%), Komparasi `M1–M3` & Uji Wilcoxon (Test 80%)

Notebook ini mengeksekusi **Tahap 4 & Tahap 5 (Hari 3)** secara ketat tanpa kebocoran data uji:
1. Memuat matriks embedding asimetris `SPECTER2` ter-normalisasi L2 (`64.183` dokumen, `96` kueri Validation, `384` kueri Held-Out Test).
2. Memvalidasi silang perhitungan mandiri `nDCG@10` dan `Recall@10` terhadap `pytrec_eval` (`max_abs_diff <= 1e-6`).
3. Menyapu `11` nilai `β ∈ {0.0, 0.1, ..., 1.0}` **khusus pada Validation Split (`20%`, `N = 96`)** untuk mengunci satu nilai optimal `β*` serta mencetak **Fig. 2** standar IEEE (`300 DPI`).
4. Mengevaluasi satu kali ketiga strategi (`M1`, `M2`, dan `M3(β*)`) pada **Held-Out Test Split (`80%`, `N = 384`)** untuk menyusun **TABLE II** dan menguji signifikansi **Paired Wilcoxon** terkoreksi Bonferroni (`α_bonf = 0.025`) untuk **TABLE III**.

In [1]:
import json
import sys
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.dual_retrieval import (
    EmbeddingStore,
    extract_topk_from_scores,
    measure_cpu_query_latency_ms,
    score_m1_positive_only,
    score_m2_naive_negation,
    score_m3_penalized_cosine,
)
from src.metrics import (
    cross_validate_with_pytrec_eval,
    evaluate_rankings_per_query,
    paired_wilcoxon_bonferroni,
)
from src.negative_benchmark import load_jsonl_records

DATA_DIR = PROJECT_ROOT / 'data' / 'litsearch'
EMBEDDINGS_DIR = PROJECT_ROOT / 'embeddings'
TABLES_DIR = PROJECT_ROOT / 'results' / 'tables'
FIGURES_DIR = PROJECT_ROOT / 'results' / 'figures'
TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

store = EmbeddingStore.load(EMBEDDINGS_DIR)
val_records = load_jsonl_records(DATA_DIR / 'negative_queries_val.jsonl')
test_records = load_jsonl_records(DATA_DIR / 'negative_queries_test.jsonl')

assert len(set(r['query_id'] for r in val_records) & set(r['query_id'] for r in test_records)) == 0
print(f'Korpus dimuat   : {store.doc_hat.shape}')
print(f'Validation Split: {len(val_records)} kueri (20.0%)')
print(f'Held-Out Test   : {len(test_records)} kueri (80.0%)')

D:\tumgas\Kuliah\Semester 5\4. METODOLOGI PENELITIAN\project\metopen-specter2-litsearch\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Korpus dimuat   : (64183, 768)
Validation Split: 96 kueri (20.0%)
Held-Out Test   : 384 kueri (80.0%)


In [2]:
# === TAHAP 5A: Kalibrasi Hyperparameter Tunggal beta* pada Validation Split (N = 96) ===
val_qids = [r['query_id'] for r in val_records]
q_pos_val, q_neg_val, q_naive_val = store.subset_vectors(val_qids)

# Evaluasi M2 pada Validation sebagai pembanding referensi
val_s_m2 = score_m2_naive_negation(q_naive_val, store.doc_hat)
val_r_m2, val_top_m2 = extract_topk_from_scores(val_s_m2, store.doc_ids, top_k=10)
val_pytrec_m2 = cross_validate_with_pytrec_eval(val_records, val_r_m2, val_top_m2, k=10)
val_eval_m2 = evaluate_rankings_per_query(val_records, val_r_m2, k=10)

beta_grid = [round(0.1 * i, 1) for i in range(11)]
sweep_rows: list[dict] = []
max_pytrec_ndcg_diff = val_pytrec_m2['max_abs_diff_ndcg@10']
max_pytrec_rec_diff = val_pytrec_m2['max_abs_diff_recall@10']

for b in beta_grid:
    s_mat = score_m3_penalized_cosine(q_pos_val, store.doc_hat, q_neg_val, beta=b)
    ranked, top_sc = extract_topk_from_scores(s_mat, store.doc_ids, top_k=10)
    pdiff = cross_validate_with_pytrec_eval(val_records, ranked, top_sc, k=10)
    max_pytrec_ndcg_diff = max(max_pytrec_ndcg_diff, pdiff['max_abs_diff_ndcg@10'])
    max_pytrec_rec_diff = max(max_pytrec_rec_diff, pdiff['max_abs_diff_recall@10'])

    ev = evaluate_rankings_per_query(val_records, ranked, k=10)
    sweep_rows.append({
        'beta': b,
        'ndcg_at_10': float(ev['ndcg@10'].mean()),
        'recall_at_10': float(ev['recall@10'].mean()),
        'vr_at_10': float(ev['vr@10'].mean()),
    })

# Kriteria pemilihan objektif bebas meta-parameter: maksimalkan nDCG@10 pada Val (tie-break: VR@10 terendah)
best_row = max(sweep_rows, key=lambda r: (round(r['ndcg_at_10'], 8), -round(r['vr_at_10'], 8)))
beta_star = float(best_row['beta'])

df_sweep_out = pd.DataFrame([
    {
        'beta': f"{r['beta']:.1f}",
        'ndcg@10': f"{r['ndcg_at_10']:.4f}",
        'recall@10': f"{r['recall_at_10']:.4f}",
        'vr@10': f"{r['vr_at_10']:.4f}",
        'is_locked_beta_star': r['beta'] == beta_star,
    }
    for r in sweep_rows
])
sweep_csv = TABLES_DIR / 'beta_sweep_val.csv'
df_sweep_out.to_csv(sweep_csv, index=False)

calibrated_cfg = {
    'beta_star': beta_star,
    'selection_criterion': 'argmax_val_ndcg@10_tiebreak_min_vr@10',
    'val_queries_count': len(val_records),
    'val_m1_metrics': {
        'ndcg@10': round(sweep_rows[0]['ndcg_at_10'], 4),
        'recall@10': round(sweep_rows[0]['recall_at_10'], 4),
        'vr@10': round(sweep_rows[0]['vr_at_10'], 4),
    },
    'val_m2_metrics': {
        'ndcg@10': round(float(val_eval_m2['ndcg@10'].mean()), 4),
        'recall@10': round(float(val_eval_m2['recall@10'].mean()), 4),
        'vr@10': round(float(val_eval_m2['vr@10'].mean()), 4),
    },
    'val_m3_beta_star_metrics': {
        'ndcg@10': round(best_row['ndcg_at_10'], 4),
        'recall@10': round(best_row['recall_at_10'], 4),
        'vr@10': round(best_row['vr_at_10'], 4),
    },
}
with (EMBEDDINGS_DIR / 'calibrated_config.json').open('w', encoding='utf-8') as f:
    json.dump(calibrated_cfg, f, indent=2)

print(f'=== HASIL SAPUAN BETA PADA VALIDATION SPLIT (N={len(val_records)}) ===')
print(df_sweep_out.to_string(index=False))
print(f'\n[LOCKED] Hyperparameter terkunci: beta* = {beta_star:.1f}')
print(f'[OK] Validasi silang pytrec_eval (Val): max_diff_ndcg={max_pytrec_ndcg_diff:.2e}, max_diff_recall={max_pytrec_rec_diff:.2e}')

=== HASIL SAPUAN BETA PADA VALIDATION SPLIT (N=96) ===
beta ndcg@10 recall@10  vr@10  is_locked_beta_star
 0.0  0.2555    0.3646 0.1625                False
 0.1  0.2553    0.3750 0.1219                False
 0.2  0.2671    0.3958 0.0823                 True
 0.3  0.2366    0.3542 0.0458                False
 0.4  0.2032    0.3177 0.0281                False
 0.5  0.1656    0.2448 0.0156                False
 0.6  0.1021    0.1667 0.0135                False
 0.7  0.0754    0.1250 0.0083                False
 0.8  0.0503    0.0938 0.0052                False
 0.9  0.0392    0.0625 0.0042                False
 1.0  0.0310    0.0521 0.0042                False

[LOCKED] Hyperparameter terkunci: beta* = 0.2
[OK] Validasi silang pytrec_eval (Val): max_diff_ndcg=0.00e+00, max_diff_recall=0.00e+00


In [3]:
# === Cetak Fig. 2 Standar IEEE Conference Paper (1 Kolom: 3.5 in, 300 DPI, Times New Roman 8 pt, Tanpa plt.title()) ===
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'mathtext.fontset': 'stix',
    'font.size': 8,
    'axes.labelsize': 8,
    'xtick.labelsize': 7,
    'ytick.labelsize': 7,
    'legend.fontsize': 7,
})

betas_np = np.array([r['beta'] for r in sweep_rows])
ndcg_np = np.array([r['ndcg_at_10'] for r in sweep_rows])
rec_np = np.array([r['recall_at_10'] for r in sweep_rows])
vr_np = np.array([r['vr_at_10'] for r in sweep_rows])

fig, ax = plt.subplots(figsize=(3.5, 2.45), dpi=300)
ax.plot(
    betas_np, rec_np,
    marker='s', markersize=3.8, linewidth=1.25,
    color='#1f618d', label='Recall@10 (↑)',
)
ax.plot(
    betas_np, ndcg_np,
    marker='o', markersize=3.8, linewidth=1.35,
    color='#117a65', label='nDCG@10 (↑)',
)
ax.plot(
    betas_np, vr_np,
    marker='^', markersize=3.8, linewidth=1.25, linestyle='--',
    color='#c0392b', label='VR@10 (↓)',
)
ax.axvline(
    beta_star,
    color='#555555',
    linestyle=':',
    linewidth=1.15,
    label=f'Locked $\\beta^* = {beta_star:.1f}$',
)

ax.set_xlabel('Negative Penalty Coefficient ($\\beta$)')
ax.set_ylabel('Validation Metric Score')
ax.set_xlim(-0.03, 1.03)
ax.set_ylim(0.0, 0.46)
ax.set_xticks(np.round(np.linspace(0.0, 1.0, 6), 1))
ax.grid(True, linestyle=':', linewidth=0.5, alpha=0.6)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(loc='upper right', frameon=True, framealpha=1.0, edgecolor='#cccccc')
fig.tight_layout()

fig2_png = FIGURES_DIR / 'fig2_beta_calibration.png'
fig2_pdf = FIGURES_DIR / 'fig2_beta_calibration.pdf'
fig.savefig(fig2_png, dpi=300, bbox_inches='tight')
fig.savefig(fig2_pdf, bbox_inches='tight')
plt.close(fig)
print(f'[OK] Fig. 2 tersimpan: {fig2_png.name} & {fig2_pdf.name}')

[OK] Fig. 2 tersimpan: fig2_beta_calibration.png & fig2_beta_calibration.pdf


In [4]:
# === TAHAP 5B: Evaluasi Satu Kali pada Held-Out Test Split (N = 384) -> TABLE II & TABLE III ===
test_qids = [r['query_id'] for r in test_records]
q_pos_test, q_neg_test, q_naive_test = store.subset_vectors(test_qids)

# 1. M1 (Standard Positive-Only, beta = 0.0)
s_test_m1 = score_m1_positive_only(q_pos_test, store.doc_hat)
r_test_m1, top_test_m1 = extract_topk_from_scores(s_test_m1, store.doc_ids, top_k=10)
diff_test_m1 = cross_validate_with_pytrec_eval(test_records, r_test_m1, top_test_m1, k=10)
ev_test_m1 = evaluate_rankings_per_query(test_records, r_test_m1, k=10)

# 2. M2 (Naive Prompt Negation: '<Q_pos>, excluding <Q_neg>')
s_test_m2 = score_m2_naive_negation(q_naive_test, store.doc_hat)
r_test_m2, top_test_m2 = extract_topk_from_scores(s_test_m2, store.doc_ids, top_k=10)
diff_test_m2 = cross_validate_with_pytrec_eval(test_records, r_test_m2, top_test_m2, k=10)
ev_test_m2 = evaluate_rankings_per_query(test_records, r_test_m2, k=10)

# 3. M3 (Proposed Dual-Input Penalized Cosine Scoring, locked beta*)
s_test_m3 = score_m3_penalized_cosine(q_pos_test, store.doc_hat, q_neg_test, beta=beta_star)
r_test_m3, top_test_m3 = extract_topk_from_scores(s_test_m3, store.doc_ids, top_k=10)
diff_test_m3 = cross_validate_with_pytrec_eval(test_records, r_test_m3, top_test_m3, k=10)
ev_test_m3 = evaluate_rankings_per_query(test_records, r_test_m3, k=10)

max_test_ndcg_diff = max(
    diff_test_m1['max_abs_diff_ndcg@10'],
    diff_test_m2['max_abs_diff_ndcg@10'],
    diff_test_m3['max_abs_diff_ndcg@10'],
)
max_test_rec_diff = max(
    diff_test_m1['max_abs_diff_recall@10'],
    diff_test_m2['max_abs_diff_recall@10'],
    diff_test_m3['max_abs_diff_recall@10'],
)

def build_summary_row(method_code: str, method_label: str, ev_dict: dict) -> dict:
    return {
        'method_code': method_code,
        'strategy': method_label,
        'n_queries': len(test_records),
        'ndcg@10_mean': f"{ev_dict['ndcg@10'].mean():.4f}",
        'ndcg@10_std': f"{ev_dict['ndcg@10'].std(ddof=1):.4f}",
        'recall@10_mean': f"{ev_dict['recall@10'].mean():.4f}",
        'recall@10_std': f"{ev_dict['recall@10'].std(ddof=1):.4f}",
        'vr@10_mean': f"{ev_dict['vr@10'].mean():.4f}",
        'vr@10_std': f"{ev_dict['vr@10'].std(ddof=1):.4f}",
    }

df_metrics_test = pd.DataFrame([
    build_summary_row('M1', 'Standard Positive-Only (beta = 0.0)', ev_test_m1),
    build_summary_row('M2', 'Naive Prompt Negation (excluding)', ev_test_m2),
    build_summary_row('M3', f'Proposed Dual-Input Penalized (beta* = {beta_star:.1f})', ev_test_m3),
])
metrics_test_csv = TABLES_DIR / 'metrics_test.csv'
df_metrics_test.to_csv(metrics_test_csv, index=False)

print(f'=== TABLE II: HASIL EVALUASI PADA HELD-OUT TEST SPLIT (N={len(test_records)}) ===')
print(df_metrics_test.to_string(index=False))
print(f'[OK] Validasi silang pytrec_eval (Test): max_diff_ndcg={max_test_ndcg_diff:.2e}, max_diff_recall={max_test_rec_diff:.2e}')

=== TABLE II: HASIL EVALUASI PADA HELD-OUT TEST SPLIT (N=384) ===
method_code                                    strategy  n_queries ndcg@10_mean ndcg@10_std recall@10_mean recall@10_std vr@10_mean vr@10_std
         M1         Standard Positive-Only (beta = 0.0)        384       0.2361      0.3537         0.3591        0.4736     0.1987    0.1459
         M2           Naive Prompt Negation (excluding)        384       0.1563      0.2868         0.2661        0.4376     0.3625    0.2510
         M3 Proposed Dual-Input Penalized (beta* = 0.2)        384       0.2378      0.3555         0.3565        0.4728     0.0828    0.0986
[OK] Validasi silang pytrec_eval (Test): max_diff_ndcg=0.00e+00, max_diff_recall=0.00e+00


In [5]:
# === Uji Signifikansi Paired Wilcoxon (Bonferroni alpha_bonf = 0.05 / 2 = 0.025) -> TABLE III ===
wilcoxon_rows: list[dict] = []
comparisons = [
    ('M3 vs. M1', ev_test_m3, ev_test_m1),
    ('M3 vs. M2', ev_test_m3, ev_test_m2),
]
for pair_label, ev_a, ev_b in comparisons:
    for metric_key, metric_display, higher_better in [
        ('ndcg@10', 'nDCG@10', True),
        ('recall@10', 'Recall@10', True),
        ('vr@10', 'VR@10', False),
    ]:
        wres = paired_wilcoxon_bonferroni(
            ev_a[metric_key],
            ev_b[metric_key],
            num_comparisons=2,
            alpha=0.05,
            higher_is_better=higher_better,
        )
        rel_pct = (
            ((wres['mean_a'] - wres['mean_b']) / wres['mean_b']) * 100.0
            if abs(wres['mean_b']) > 1e-12 else 0.0
        )
        wilcoxon_rows.append({
            'comparison': pair_label,
            'metric': metric_display,
            'mean_diff': f"{wres['mean_diff']:+.4f}",
            'rel_change_pct': f"{rel_pct:+.2f}%",
            'improved_tied_degraded': f"{wres['improved']} / {wres['ties']} / {wres['degraded']}",
            'wilcoxon_stat': f"{wres['wilcoxon_stat']:.1f}",
            'p_raw': f"{wres['p_raw']:.3e}",
            'p_bonf': f"{wres['p_bonf']:.3e}",
            'effect_size_r': f"{wres['effect_size_r']:.4f}",
            'significant_at_0.025': wres['significant_bonf'],
        })

df_wilcoxon = pd.DataFrame(wilcoxon_rows)
wilcoxon_csv = TABLES_DIR / 'wilcoxon.csv'
df_wilcoxon.to_csv(wilcoxon_csv, index=False)

# Simpan hasil per-kueri Test Split untuk Error Analysis Hari 4 + ukur latensi CPU
per_query_path = TABLES_DIR / 'per_query_test_results.jsonl'
with per_query_path.open('w', encoding='utf-8') as f:
    for idx, rec in enumerate(test_records):
        row_out = {
            'query_id': rec['query_id'],
            'q_pos': rec['q_pos'],
            'q_neg': rec['q_neg'],
            'q_naive_not': rec['q_naive_not'],
            'specificity': rec['specificity'],
            'query_set': rec['query_set'],
            'gold_ids': rec['gold_ids'],
            'violator_count': rec['violator_count'],
            'm1_top10': r_test_m1[idx],
            'm2_top10': r_test_m2[idx],
            'm3_top10': r_test_m3[idx],
            'm1_top10_scores': [round(float(x), 6) for x in top_test_m1[idx]],
            'm2_top10_scores': [round(float(x), 6) for x in top_test_m2[idx]],
            'm3_top10_scores': [round(float(x), 6) for x in top_test_m3[idx]],
            'm1_ndcg10': round(float(ev_test_m1['ndcg@10'][idx]), 6),
            'm2_ndcg10': round(float(ev_test_m2['ndcg@10'][idx]), 6),
            'm3_ndcg10': round(float(ev_test_m3['ndcg@10'][idx]), 6),
            'm1_recall10': round(float(ev_test_m1['recall@10'][idx]), 6),
            'm2_recall10': round(float(ev_test_m2['recall@10'][idx]), 6),
            'm3_recall10': round(float(ev_test_m3['recall@10'][idx]), 6),
            'm1_vr10': round(float(ev_test_m1['vr@10'][idx]), 6),
            'm2_vr10': round(float(ev_test_m2['vr@10'][idx]), 6),
            'm3_vr10': round(float(ev_test_m3['vr@10'][idx]), 6),
        }
        f.write(json.dumps(row_out, ensure_ascii=False) + '\n')

cpu_latency = measure_cpu_query_latency_ms(store, test_qids[:50], beta=beta_star, top_k=10)
summary_day3 = {
    'calibrated_beta_star': beta_star,
    'val_queries': len(val_records),
    'test_queries': len(test_records),
    'pytrec_eval_max_abs_diff': {
        'val_ndcg@10': max_pytrec_ndcg_diff,
        'val_recall@10': max_pytrec_rec_diff,
        'test_ndcg@10': max_test_ndcg_diff,
        'test_recall@10': max_test_rec_diff,
    },
    'cpu_single_query_latency_ms': cpu_latency,
    'test_metrics': df_metrics_test.to_dict(orient='records'),
    'wilcoxon_tests': wilcoxon_rows,
}
with (TABLES_DIR / 'day3_evaluation_summary.json').open('w', encoding='utf-8') as f:
    json.dump(summary_day3, f, indent=2)

print('=== TABLE III: UJI PAIRED WILCOXON SIGNED-RANK (BONFERRONI alpha = 0.025) ===')
print(df_wilcoxon.to_string(index=False))
print(f"\n[INFO] Latensi pencarian CPU per kueri (64.183 dok): {cpu_latency['mean_ms']} +/- {cpu_latency['std_ms']} ms (median {cpu_latency['median_ms']} ms)")

=== TABLE III: UJI PAIRED WILCOXON SIGNED-RANK (BONFERRONI alpha = 0.025) ===
comparison    metric mean_diff rel_change_pct improved_tied_degraded wilcoxon_stat     p_raw    p_bonf effect_size_r  significant_at_0.025
 M3 vs. M1   nDCG@10   +0.0016         +0.69%          56 / 289 / 39        2096.5 4.956e-01 9.911e-01        0.0348                 False
 M3 vs. M1 Recall@10   -0.0026         -0.73%          10 / 363 / 11         110.0 8.273e-01 1.000e+00        0.0111                 False
 M3 vs. M1     VR@10   -0.1159        -58.32%          265 / 119 / 0           0.0 5.092e-47 1.018e-46        0.7349                  True
 M3 vs. M2   nDCG@10   +0.0815        +52.16%          96 / 264 / 24         975.5 3.552e-12 7.103e-12        0.3549                  True
 M3 vs. M2 Recall@10   +0.0904        +33.95%           46 / 329 / 9         270.0 3.069e-06 6.137e-06        0.2381                  True
 M3 vs. M2     VR@10   -0.2797        -77.16%           320 / 64 / 0           0.0 1.962